Resultaten wegschrijven in genormaliseerd datamodel. Voorlopig gesimuleerd als sqlite. Connectie met LSVI databank of andere masterdata nog uit te klaren?

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import sqlite3
from datetime import datetime
import pandas as pd
import geopandas as gpd

from arcgis.gis import GIS
from arcgis.features import FeatureLayer

import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

In [31]:
pd.set_option('display.max_columns', None)  

### Connectie naar databank

In [32]:
# Aanmaken 
# def init_database(db_path="lsvi_resultaten_slank.sqlite"):
#     conn = sqlite3.connect(db_path)
#     cursor = conn.cursor()
#     cursor.execute("PRAGMA foreign_keys = ON;")
    
#     # Tabel 1: WaarnemingEvent
#     cursor.execute("""
#     CREATE TABLE IF NOT EXISTS waarneming_event (
#         collectie_id TEXT PRIMARY KEY,
#         global_id TEXT,
#         user_name TEXT,
#         bwk_plot_id TEXT,
#         bwk_globalid TEXT,
#         bwk_centroid_x REAL,
#         bwk_centroid_y REAL,
#         locatie_x REAL,
#         locatie_y REAL,
#         EPSG INTEGER,
#         doel_habitattype TEXT,
#         created_date DATETIME,
#         last_edited_date DATETIME,
#         timestamp_measurement DATETIME
#     );
#     """)
    
#     # Tabel 2: Resultaat (Nu ook voor de losse soorten!)
#     cursor.execute("""
#     CREATE TABLE IF NOT EXISTS resultaat (
#         resultaat_id INTEGER PRIMARY KEY AUTOINCREMENT,
#         collectie_id TEXT,
#         voorwaarde_id INTEGER,
#         vraag_id TEXT,
#         subvraag TEXT,
#         waarde_tekst TEXT,
#         waarde_numeriek REAL,
#         FOREIGN KEY (collectie_id) REFERENCES waarneming_event(collectie_id) ON DELETE CASCADE
#     );
#     """)
#     conn.commit()
#     return conn

# Helperfunctie om ArcGIS Epoch Milliseconds om te zetten naar een ISO string
def format_agol_date(epoch_ms):
    if epoch_ms is None:
        return None
    # AGOL timestamps zijn in milliseconden, Python verwacht seconden
    return datetime.fromtimestamp(epoch_ms / 1000.0).strftime('%Y-%m-%d %H:%M:%S')



### Run ETL

In [33]:
# Uitvoering
# Joost to do: credentials uit key vault of environment
# Load secrets into environment
LOCAL_DB = "G:\\Mijn Drive\\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"
AGOL_URL = "https://gisservices.inbo.be/portal"

AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(
        db_path=LOCAL_DB, entry_name=ENTRY_TITLE
    )

today = datetime.now().strftime("%Y%m%d")
db_path = f"../output/lsvi_resultaten_{today}.sqlite"

# Verbinden met AGOL
print(f"Verbinden met {AGOL_URL}...")
gis = GIS(AGOL_URL, AGOL_USER, AGOL_PASS)

# Not used because simplified export to CSV
# conn = init_database(db_path)
# cursor = conn.cursor()

feature_layer_list = ['xlsform_hab1-4', 'xlsform_hab5-7', 'xlsform_hab9']

ℹ️ Inloggegevens voor 'AGOL' zijn al actief in deze sessie. Prompt overgeslagen.
Verbinden met https://gisservices.inbo.be/portal...


### Backup feature layer in AGOL

In [91]:
for feature_layer in feature_layer_list:
    # Lookup feature layer ID based on name
    layer_items = gis.content.search(feature_layer, item_type='Feature Layer', max_items=10)
    # Exclude item in list if ends with _form
    layer_items = [item for item in layer_items if not item.title.endswith('_form')]
    feature_layer_item_id = layer_items[0].id if layer_items else None
    fl_item = gis.content.get(feature_layer_item_id)

    # Export to GeoPackage (created in your root Enterprise content by default)
    backup_title = f"{fl_item.title}_backup_{today}"
    exported_item = fl_item.export(
        title=backup_title,
        export_format="GeoPackage",
        wait=True
    )

    # Move the exported GeoPackage item to a specific user folder
    target_folder_name = "Survey-LSVI Backup"

    # Check if folder exists; if not, create it
    existing_folders = [f.name for f in gis.users.me.folders]
    if target_folder_name not in existing_folders:
        gis.content.folders.create(target_folder_name)

    # Delete existing backup in the target folder if it already exists
    folder_items = gis.users.me.items(folder=target_folder_name)
    for old_item in folder_items:
        if old_item.title == backup_title:
            print(f"Deleting existing backup item '{old_item.title}' in '{target_folder_name}'...")
            old_item.delete()

    # Move item to target folder
    exported_item.move(target_folder_name)

    print(f"Exported item '{exported_item.title}' moved to folder '{target_folder_name}'.")

Deleting existing backup item 'xlsform_hab1-4_backup_20260917' in 'Survey-LSVI Backup'...
Exported item 'xlsform_hab1-4_backup_20260917' moved to folder 'Survey-LSVI Backup'.
Deleting existing backup item 'xlsform_hab5-7_backup_20260917' in 'Survey-LSVI Backup'...
Exported item 'xlsform_hab5-7_backup_20260917' moved to folder 'Survey-LSVI Backup'.
Deleting existing backup item 'xlsform_hab9_backup_20260917' in 'Survey-LSVI Backup'...
Exported item 'xlsform_hab9_backup_20260917' moved to folder 'Survey-LSVI Backup'.


### Run ETL 
Feature Layer --> CSV

This CSV is an overview of all questions and answers linked to a single survey. 

In [92]:
# Initialize rows list for output
rows = []

# Loop over feature layers
for FEATURE_LAYER_NAME in feature_layer_list:
    # Get feature layer id from name
    layer_items = gis.content.search(FEATURE_LAYER_NAME, item_type='Feature Layer', max_items=10)
    # Exclude item in list if ends with _form
    layer_items = [item for item in layer_items if not item.title.endswith('_form')]
    feature_layer_item_id = layer_items[0].id if layer_items else None
    print(f"Feature layer {feature_layer_item_id} downloaden...")
    feature_layer = layer_items[0].layers[0]

    # Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
    features_result = feature_layer.query(where="1=1", out_sr=4326)
    print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

    # Field aliases contain more information about question and their labels
    field_aliases = {
        f.name: f.alias for f in feature_layer.properties.fields
    } if hasattr(feature_layer, 'properties') else {}

    # Loop over each feature in feature layer
    for feature in features_result.features:
        geom = feature.geometry if feature.geometry else {}
        attrs = feature.attributes if feature.attributes else {}
        
        # Controleer of de cruciale collectie_id aanwezig is
        collectie_id = attrs.get('collectie_id')
        if not collectie_id:
            collectie_id = attrs.get('globalid')
            
        # Metadata extraheren en parsen
        global_id = attrs.get('globalid')
        created_user = attrs.get('created_user')
        last_edited_user = attrs.get('last_edited_user')
        habitat_keuze = attrs.get('habitat_keuze')
        bwk_plot_id = attrs.get('bwk_plot_id')
        bwk_globalid = attrs.get('bwk_globalid')
        bwk_centroid_x = attrs.get('bwk_centroid_x')
        bwk_centroid_y = attrs.get('bwk_centroid_y')
        
        created_date_txt = format_agol_date(attrs.get('datum'))
        last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
        
        # Tijdstip van het bezoek bepalen (Datum-veld + Uur-veld combineren)
        datum_txt = format_agol_date(attrs.get('datum'))
        uur_txt = attrs.get('uur')  # string zoals "14:59"
        tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

        # Opmerkingen
        opmerkingen = attrs.get('opmerkingen')

        # Niet grondig doorzocht
        niet_grondig_doorzocht = attrs.get(f"niet_grondig_{habitat_keuze}")

        # Geometrie
        x = geom.get('x')
        y = geom.get('y')
        epsg = geom.get('spatialReference', {}).get('wkid')

        metadata = {
                    'collectie_id': collectie_id,
                    'global_id': global_id,
                    'created_user': created_user,
                    'last_edited_user': last_edited_user,
                    'bwk_plot_id': bwk_plot_id,
                    'bwk_globalid': bwk_globalid,
                    'bwk_centroid_x_l72': bwk_centroid_x,
                    'bwk_centroid_y_l72': bwk_centroid_y,
                    'locatie_x_wgs84': x,
                    'locatie_y_wgs84': y,
                    'epsg': epsg,
                    'doel_habitattype': habitat_keuze,
                    'created_date': created_date_txt,
                    'last_edited_date': last_edited_date_txt,
                    'timestamp_measurement': tijdstip_waarneming,
                    'opmerkingen': opmerkingen,
                    'niet_grondig_doorzocht': niet_grondig_doorzocht
                }
        
        # Loop dynamisch door alle kolommen van de feature
        for key, value in attrs.items():
            # Sla lege cellen, systeemenmerken en layout-hulpmiddelen over
            if value is None:
                continue

            # Extraheer het getal (VoorwaardeID) als de kolom begint met 'vrg_'
            voorwaarde_id = None
            habitattype = None
            subvraag = None

            if key.startswith("vrg_"):
                clean_key = key[4:]

                # Controleer of het een matrixvraag betreft
                if "_matrix_" in clean_key:
                    # Bv. '736_1310_zv_matrix_0' -> base_part = '736_1310_zv'
                    base_part, groep_id = clean_key.split("_matrix_") # if groep = Sleutelsoorten --> groep_id is taxon_id
                    # Ophalen van de weergavenaam (subvraag) uit onze lookup dict
                    subvraag = field_aliases.get(key)
                else:
                    base_part = clean_key
                    groep_id = None
                    subvraag = None  # Geen matrix, dus geen subvraag

                # Ontleed voorwaarde_id en habitattype uit base_part (bv. '712_1310_zk')
                parts = base_part.split("_", 1)
                voorwaarde_id = int(parts[0]) if parts[0].isdigit() else None
                habitattype = parts[1] if len(parts) > 1 else None
            
                # Gegevenstype bepalen voor waarde_numeriek
                waarde_numeriek = None
                if isinstance(value, (int, float)):
                    waarde_numeriek = float(value)
                elif isinstance(value, str):
                    try:
                        waarde_numeriek = float(value)
                    except ValueError:
                        pass

                # Afhandeling van select_multiple (komma-gescheiden waarden)
                if isinstance(value, str) and "," in value:
                    # Comma-separated → per soort 1 rij
                    soorten = [s.strip() for s in value.split(",")]
                    for soort in soorten:
                        if soort:
                            row = {
                                **metadata,  # Unpack metadata
                                'voorwaarde_id': voorwaarde_id,
                                'habitattype': habitattype,
                                'groep_id': groep_id,
                                'subvraag': subvraag,
                                'vraag_id': key,
                                'waarde_tekst': soort,
                                'waarde_numeriek': None
                            }
                            rows.append(row)
                else:
                    # Enkele waarde (numeriek of tekst)
                    row = {
                        **metadata,  # Unpack metadata
                        'voorwaarde_id': voorwaarde_id,
                        'habitattype': habitattype,
                        'groep_id': groep_id,
                        'subvraag': subvraag,
                        'vraag_id': key,
                        'waarde_tekst': str(value) if not waarde_numeriek else None,
                        'waarde_numeriek': waarde_numeriek
                    }
                    rows.append(row)
                    
# Eind van loop
df_resultaat = pd.DataFrame(rows)  
print("Export afgerond. De resultaten zijn opgeslagen in een DataFrame.")

Feature layer fadd6d9740ca4dd0bac2bf937038775c downloaden...
Succesvol 19 features gedownload. Start verwerking...
Feature layer b126bc2717714080b2a1adc3a1c5ddfa downloaden...
Succesvol 7 features gedownload. Start verwerking...
Feature layer 23e7fcfd9e9748b39e140cfda6cd3a63 downloaden...
Succesvol 6 features gedownload. Start verwerking...
Export afgerond. De resultaten zijn opgeslagen in een DataFrame.


In [93]:
# Remove centerpoint for now because not correct
df_resultaat.drop(columns=['bwk_centroid_x_l72', 'bwk_centroid_y_l72'], inplace=True)
df_resultaat.to_csv(f"../output/lsvi_resultaten_{today}.csv", index=False, encoding='utf-8-sig')

### Fetch latest BWK information
Export to geopackage. Does not contain question/answer information.
Only survey metadata (which survey was taken where, linked to which BWK polygon)

Geometry of BWK polygon can be changed compared to field work.
Also distribution of habitat types can be changed.
LSVI information is still relevant, but in export we prefer to show the latest BWK information.

In [94]:
# Get BWK feature layer from field maps
url = "https://gisservices.inbo.be/arcgis/rest/services/Veld/Veld_BWKhab2026/FeatureServer/0"
layer = FeatureLayer(url)

# Get list of global IDs
unique_gids = df_resultaat.bwk_globalid.unique()
formatted_ids = ",".join([f"'{str(gid).strip('{}').upper()}'" for gid in unique_gids if pd.notna(gid)])
where_clause = f"GlobalID IN ({formatted_ids})"

# Query the layer (return_geometry=True fetches the polygons)
# out_fields restricts the downloaded attributes to save bandwidth
out_fields = ["GlobalID", 'HAB1', 'HAB2', 'HAB3', "pHAB1", "pHAB2", "pHAB3"]
feature_set = layer.query(where=where_clause, out_fields=out_fields, return_geometry=True, out_sr=31370)
bwk_orig = feature_set.sdf

# Extract center point of polygons
# Extract numerical x and y from the Shapely centerpoint geometry
bwk_orig['centerpoint'] = bwk_orig['SHAPE'].apply(lambda geom: geom.centroid if geom else None)
bwk_orig['bwk_centroid_x_l72'] = bwk_orig['centerpoint'].apply(lambda pt: pt[0] if pt is not None else None)
bwk_orig['bwk_centroid_y_l72'] = bwk_orig['centerpoint'].apply(lambda pt: pt[1] if pt is not None else None)

# Create geodataframe from the BWK original data
bwk_orig_gdf = gpd.GeoDataFrame(bwk_orig, geometry='SHAPE', crs="EPSG:31370")

bwk_orig_gdf['bwk_globalid'] = bwk_orig_gdf['GlobalID']
bwk_orig_gdf.drop(columns=['OBJECTID','GlobalID','centerpoint'], inplace=True)

# Unpivot BWK information
bwk_orig_gdf_melted = pd.wide_to_long(
    bwk_orig_gdf,
    stubnames=['HAB', 'pHAB'],
    i=['bwk_globalid', 'bwk_centroid_x_l72', 'bwk_centroid_y_l72'],
    j='hab_num'
).reset_index()

# Remove na and empty habitat types
bwk_orig_gdf_melted = bwk_orig_gdf_melted.dropna(subset=['HAB'])
bwk_orig_gdf_melted = bwk_orig_gdf_melted[bwk_orig_gdf_melted['HAB'] != '']
bwk_orig_gdf_melted = bwk_orig_gdf_melted[bwk_orig_gdf_melted['pHAB'] != 0]

In [95]:
# Link to survey metadata

# Only keep survey metadata
keep_cols = ['collectie_id','global_id','created_user','last_edited_user','bwk_plot_id','bwk_globalid', 'locatie_x_wgs84', 'locatie_y_wgs84', 'doel_habitattype','created_date','last_edited_date','opmerkingen','niet_grondig_doorzocht']
df_resultaat_small = df_resultaat[keep_cols]
df_resultaat_small = df_resultaat_small.drop_duplicates()
print(df_resultaat_small.shape)

# Merge correct centerpoint and poly geometry back to results
# df_resultaat.drop(columns=['bwk_centroid_x_l72', 'bwk_centroid_y_l72'], inplace=True)
df_merged = df_resultaat_small.merge(
    bwk_orig_gdf_melted,
    how='left',
    left_on=['bwk_globalid', 'doel_habitattype'],
    right_on=['bwk_globalid', 'HAB'],
)

# Make geodataframe (bwk polygon as geometry)
gdf_resultaat = gpd.GeoDataFrame(df_merged, geometry='SHAPE', crs="EPSG:31370")
print(gdf_resultaat.shape)

(31, 13)
(31, 19)


In [96]:
# Write to geopackage
gdf_surveys = gdf_resultaat[keep_cols + ['HAB','pHAB','bwk_centroid_x_l72', 'bwk_centroid_y_l72', 'SHAPE']]
# Drop duplicates
gdf_surveys = gdf_surveys.drop_duplicates()
gdf_surveys.shape
gdf_surveys.to_file(f"../output/lsvi_surveys_{today}.gpkg", driver="GPKG")

In [97]:
len(gdf_surveys.collectie_id.unique())

31

- 1 collectie_id is 1 survey. 
- Deze collectie is gelink aan een plot_ID (bwk laag) indien ingevuld, maar ook via de globalid van BWK en centroide van polygoon
- We houden zowel deze centroide bij (L72) als de locatie van het device bij openen van de survey (WGS84).
    De centroide coordinaten hebben een decimaal punt ('.') hetgeen automatisch verwijderd wordt bij opslag in databank wegens regio/taal settings. Workaround moet nog getest worden bij nieuwe versie van de survey.
- De voorwaarde id komt overeen met voorwaarde id uit invoervereisten
- Indien 1 voorwaarde (vraag), 1 antwoord kent, zal je het antwoord in kolom waarde_tekst of waarde_numeriek terugvinden.
- In geval van select multiple, gaan de meerdere antwoorden gesplitst worden over meerdere rijen (met dezelfde collectieID en voorwaarde ID)
- In geval van matrixvraag: voor eenzelfde voorwaarde id ga je meerdere rijen terugvinden met een groep_id ingevuld en bijhorende subvraag. Indien de matrixvraag voor sleutelsoorten is, zal de groep_id de taxon_id zijn uit gekoppelde soortenlijst en subvraag de nederlandstalige naam van deze soort. Indien de matrixvraag een andere groepering kent, zal de groep_id verwijzen naar de gegeven groep in de invoervereisten (bv. groeiklasse 1 in groep van 'groeiklassen bomen' kent id = 21).
- Veldmedewerker kan 'ja' of 'nee' aanduiden bij de vraag 'Niet grondig onderzocht'. ja --> snel ingevuld maar niet volledig onderzocht. Een extra veld met opmerkingen is ook mogelijk.